# 11 - A2: frozen SSL + learned layer-weighted head

**System A2** (README.md "The plan", step 3): a frozen SSL backbone (WavLM Base+, then
XLS-R 300M) produces per-layer hidden states; the only trainable part is a small head
(`audio_df.models.ssl_head.SSLHead`) that learns a softmax weight per layer, sums the
layers, LayerNorms, projects, and runs a small MLP to one logit per clip (higher = more
bona fide, matching this project's `higher_is_spoof=False` score convention).

## The pooled-cache equivalence this design rests on

Caching every frame of every layer for the whole corpus is the "one real blowup"
`tasks/CONVENTIONS.md` names explicitly: 25 layers x 199 frames x 1024 dims x 2 bytes (fp16)
per clip is ~10.2 MB, and this corpus has 121,461 clips -> ~1.24 TB, infeasible on a 16 GB
fanless machine. Instead `audio_df.ssl_cache.py` caches only the **time-mean** of each
layer's hidden states: `[n_layers, hidden_dim]` fp16 per clip (~51.2 KB at XLS-R's size,
smaller still for WavLM Base+'s 13 x 768) -- about 6.2 GB for the whole corpus at XLS-R
size, ~2.4 GB per silence mode at WavLM Base+ size.

This is not a lossy shortcut. `SSLHead`'s forward path computes a **weighted sum over
layers**, then (implicitly, since the cache already is the per-layer time-mean) a **mean
over time**. Both operations are linear, and linear operators commute:

```
sum_l w_l * mean_t(H^(l))  ==  mean_t(sum_l w_l * H^(l))
```

So `SSLHead(pooled_cache)` is *exactly* equal (up to fp16 rounding) to computing the
weighted sum on the full frame-level hidden states and pooling afterwards -- never an
approximation. `tests/test_ssl_cache.py::test_pooled_cache_equals_weighted_sum_then_pool_on_full_frames`
and `tests/test_ssl_head.py::test_weighted_sum_matches_frame_level_weighted_sum_then_mean_pool`
both check this numerically, from the extraction side and the head side respectively. This
is why extraction pools *before* any LayerNorm (LayerNorm is not linear and would break the
equivalence) -- `SSLHead`'s LayerNorm/projection/MLP are applied strictly after the
already-pooled weighted sum.

## Fixed design decisions (see `run_a2.py`'s own module docstring for the full list)

- **Center crop for every split, including train** -- a fixed on-disk cache cannot support
  per-epoch random-crop augmentation.
- **No trimmed training, ever** -- `train` always uses the raw cache; `trimmed` is a
  diagnostic applied only when *scoring* eval/In-the-Wild with the raw-trained head.
- **Checkpoint selection is by dev LOSS, not dev EER** -- dev only contains attacks
  A01-A06, so dev EER saturates and stops being a useful early-stopping signal.
- **Per-backbone input normalization** -- WavLM Base+'s own feature extractor has
  `do_normalize=false`; XLS-R 300M's has `do_normalize=true`
  (`research/knowledge/methods/ssl-hidden-states.md`). `audio_df.ssl_cache.load_backbone`
  always loads each model's own `AutoFeatureExtractor`, never a hardcoded shared choice.

This notebook holds **questions, plots and interpretation only** -- every reusable piece of
logic lives in `audio_df.ssl_cache`, `audio_df.models.ssl_head`, and
`audio_df.experiments.run_a2`.

## Config, resource limits, and current memory status

Loads the Base+ config (`configs/a2_wavlm_base_plus.yaml` by default; swap in
`configs/a2_xlsr_300m.yaml` to inspect the XLS-R config -- **run XLS-R only after Base+ works
end to end, and raw-only**, per that config's own header comment) and applies this
machine's resource limits (`configs/resources.yaml`) the same way every runner does, so
this notebook's cells reflect the same environment a real run would.

In [ ]:
from pathlib import Path

from audio_df.experiments import run_a2
from audio_df.resources import apply_resource_limits, load_resources, memory_status

cfg = run_a2.load_config("configs/a2_wavlm_base_plus.yaml")
res = load_resources()
apply_resource_limits(res)

print("model:", cfg["model"])
print("preprocessing:", cfg["preprocessing"])
print("head:", cfg["head"])
print()
print("current memory status:", memory_status())

## Timing run, and how to read it

`run_a2.py timing` extracts pooled hidden states for `configs/resources.yaml:
timing_run_clips` (200) train clips into a throwaway location, prints
`audio_df.resources.format_timing` (median/mean/p90 ms per item, items/sec, peak RSS, peak
MPS allocation, swap growth, memory pressure before/after) and then extrapolates -- as a
**lower bound, not a promise** (this is a fanless machine that throttles under sustained
load) -- to:

- all raw splits (train+dev+eval, 121,461 clips),
- trimmed eval (71,237 clips),
- In-the-Wild (~31,800 clips -- **unverified count**, per
  `research/fact-checks/FC-plan-2026-09-30.md` item 3: only aggregate hours were confirmed
  against a primary source, not a clip count),
- the E4 robustness subset (4 degradations x 10k clips).

It writes `data/cache/timing/a2_<model_tag>.json` and creates **no** experiment directory
and **no** cache -- it is a sizing check, run before committing to a real `extract`.

This notebook does not run `timing` itself (that means downloading and running the real
model, forbidden for this agent per its brief) -- run it from the terminal:

```bash
uv run python -m audio_df.experiments.run_a2 timing --config configs/a2_wavlm_base_plus.yaml
```

then re-load and inspect the JSON it wrote, e.g.:

```python
import json
result = json.loads(Path("data/cache/timing/a2_wavlm-base-plus.json").read_text())
result["timing"], result["extrapolations_hours"]
```

In [ ]:
import json

timing_path = Path("data/cache/timing/a2_wavlm-base-plus.json")
if timing_path.exists():
    timing_result = json.loads(timing_path.read_text())
    print(json.dumps(timing_result, indent=2))
else:
    print(f"{timing_path} not found yet -- run the `timing` command from the terminal first "
          "(see the markdown cell above).")

## The exact terminal commands, in order

Run every heavy command with `caffeinate -i` so the machine does not sleep mid-job; every
command is resumable (`--resume`) if interrupted or if `MemoryPressureExit` fires (exit code
3). Expected times come from the `timing` JSON above, not guessed.

```bash
# 1. sizing check (writes data/cache/timing/a2_wavlm-base-plus.json, no cache/experiment)
uv run python -m audio_df.experiments.run_a2 timing --config configs/a2_wavlm_base_plus.yaml

# 2. build the raw pooled cache for train, dev, eval
caffeinate -i uv run python -m audio_df.experiments.run_a2 extract \
    --config configs/a2_wavlm_base_plus.yaml --testsets train,dev,eval --silence-mode raw

# 3. train the head (raw only; selects by dev loss; scores dev+eval raw)
caffeinate -i uv run python -m audio_df.experiments.run_a2 train \
    --config configs/a2_wavlm_base_plus.yaml --exp e1

# 4. trimmed diagnostic: extract ONLY eval (and itw, once its manifest exists) under
#    silence_mode=trimmed -- there is no trimmed train/dev cache, by design
caffeinate -i uv run python -m audio_df.experiments.run_a2 extract \
    --config configs/a2_wavlm_base_plus.yaml --testsets eval,itw --silence-mode trimmed

# 5. score the raw-trained head against the trimmed eval/itw caches
uv run python -m audio_df.experiments.run_a2 score \
    --config configs/a2_wavlm_base_plus.yaml \
    --model-from experiments/E-<today>-e1-a2-raw --exp e1 \
    --silence-mode trimmed --testsets eval,itw

# 6. metrics -- ALWAYS from score_experiment.py, never computed by the runner
uv run python scripts/score_experiment.py experiments/E-<today>-e1-a2-raw
uv run python scripts/score_experiment.py experiments/E-<today>-e1-a2-trimmed \
    --threshold-from experiments/E-<today>-e1-a2-raw
uv run python scripts/make_tables.py
```

Each `run_a2.py` subcommand prints the exact next command at the end -- copy that rather
than retyping the above, since it fills in the real experiment directory path.

## After training: layer weights, loss curves, score distributions

Once `train` has produced `experiments/E-<today>-e1-a2-raw/{layer_weights.json,
training_curve.json, scores/{dev,eval}.csv}` and `scripts/score_experiment.py` has written
`metrics.json`, load and plot them here. Set `exp_dir` to the real experiment path.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

exp_dir = Path("experiments/E-20261001-e1-a2-raw")  # <- set to the real run

if exp_dir.exists():
    layer_weights = json.loads((exp_dir / "layer_weights.json").read_text())
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(range(len(layer_weights)), layer_weights)
    ax.set_xlabel("layer index (0 = embedding output)")
    ax.set_ylabel("learned softmax weight")
    ax.set_title(f"{exp_dir.name}: learned layer weights")
    plt.show()
else:
    print(f"{exp_dir} does not exist yet -- run `train` first, then update exp_dir above.")

In [ ]:
if exp_dir.exists():
    curve = json.loads((exp_dir / "training_curve.json").read_text())
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(curve["train_loss"], label="train")
    ax.plot(curve["dev_loss"], label="dev")
    ax.axvline(curve["best_epoch"], color="grey", linestyle="--", label="best (dev loss)")
    ax.set_xlabel("epoch")
    ax.set_ylabel("BCE loss")
    ax.legend()
    ax.set_title(f"{exp_dir.name}: train/dev loss (pos_weight={curve['pos_weight']:.2f})")
    plt.show()

In [ ]:
if exp_dir.exists():
    for testset in ("dev", "eval"):
        scores_path = exp_dir / "scores" / f"{testset}.csv"
        if not scores_path.exists():
            continue
        df = pd.read_csv(scores_path)
        fig, ax = plt.subplots(figsize=(6, 3))
        for label, group in df.groupby("label"):
            ax.hist(group["score"], bins=40, alpha=0.6, label=label)
        ax.set_xlabel("score (higher = more bona fide)")
        ax.legend()
        ax.set_title(f"{exp_dir.name}: {testset} score distribution by class")
        plt.show()

        if "attack_id" in df.columns:
            fig, ax = plt.subplots(figsize=(8, 3))
            df[df["label"] == "spoof"].boxplot(column="score", by="attack_id", ax=ax)
            ax.set_title(f"{exp_dir.name}: {testset} spoof scores by attack_id")
            plt.show()

In [ ]:
metrics_path = exp_dir / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print(json.dumps(metrics, indent=2))
else:
    print(f"{metrics_path} not found -- run "
          f"`uv run python scripts/score_experiment.py {exp_dir}` first.")